# Analysis 09 worst site rankings

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# Analysis 09: Worst-Site Rankings

Rank the worst-performing test sites for `hs`, `tp`, `dir`, and `dp`, then build an overall worst-site ranking that combines all four targets using percentile-normalized RMSE scores.


In [ ]:
from pathlib import Path
import warnings
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "training.yaml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing configs/training.yaml and src/")


REPO_ROOT = find_repo_root()
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"

from notebooks.multisource_notebook_helpers import load_prediction_table  # noqa: E402

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)


def show_note(message: str) -> None:
    display(Markdown(message))


def require_columns(df: pd.DataFrame, columns: list[str], label: str) -> None:
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise KeyError(f"{label} is missing required columns: {missing}")


def angle_from_sin_cos(sin_values: Any, cos_values: Any) -> np.ndarray:
    sin_arr = np.asarray(sin_values, dtype=float)
    cos_arr = np.asarray(cos_values, dtype=float)
    return np.mod(np.degrees(np.arctan2(sin_arr, cos_arr)), 360.0)


def wrap_angle_signed_deg(values: Any) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    return ((arr + 180.0) % 360.0) - 180.0


def circular_error_deg(pred_deg: Any, true_deg: Any, absolute: bool = True) -> np.ndarray:
    delta = wrap_angle_signed_deg(
        np.asarray(pred_deg, dtype=float) - np.asarray(true_deg, dtype=float)
    )
    return np.abs(delta) if absolute else delta


def circular_rmse_deg(pred_deg: Any, true_deg: Any) -> float:
    err = circular_error_deg(pred_deg, true_deg, absolute=False)
    err = err[np.isfinite(err)]
    if err.size == 0:
        return float("nan")
    return float(np.sqrt(np.mean(err**2)))


def rmse(y_true: Any, y_pred: Any) -> float:
    true_arr = np.asarray(y_true, dtype=float)
    pred_arr = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(true_arr) & np.isfinite(pred_arr)
    if not np.any(mask):
        return float("nan")
    return float(np.sqrt(np.mean((pred_arr[mask] - true_arr[mask]) ** 2)))


def load_optional_target_metrics_summary(results_dir: str | Path) -> pd.DataFrame | None:
    path = Path(results_dir)
    if not path.is_absolute():
        path = (REPO_ROOT / path).resolve()
    summary_path = path / "target_metrics_summary.csv"
    if not summary_path.exists():
        return None
    return pd.read_csv(summary_path)

In [ ]:
RESULTS_DIR: str | Path = "results/FINAL_RESULTS_V2/18_cnn5chan_v1"
TOP_N_SITES: int = 10
SPLIT: str = "test"
OVERALL_METHOD: str = "percentile_mean"

In [ ]:
if SPLIT != "test":
    raise ValueError("This notebook is intended for test predictions, so SPLIT must be 'test'.")
if OVERALL_METHOD != "percentile_mean":
    raise ValueError("Only OVERALL_METHOD='percentile_mean' is supported in this notebook.")

required_prediction_columns = [
    "site",
    "target_hs",
    "pred_hs",
    "target_tp",
    "pred_tp",
    "target_dir_sin",
    "pred_dir_sin",
    "target_dir_cos",
    "pred_dir_cos",
    "target_dp_sin",
    "pred_dp_sin",
    "target_dp_cos",
    "pred_dp_cos",
]

predictions_df = load_prediction_table(RESULTS_DIR, split=SPLIT).copy()
require_columns(predictions_df, required_prediction_columns, "Prediction table")

predictions_df["target_dir_deg"] = angle_from_sin_cos(
    predictions_df["target_dir_sin"], predictions_df["target_dir_cos"]
)
predictions_df["pred_dir_deg"] = angle_from_sin_cos(
    predictions_df["pred_dir_sin"], predictions_df["pred_dir_cos"]
)
predictions_df["target_dp_deg"] = angle_from_sin_cos(
    predictions_df["target_dp_sin"], predictions_df["target_dp_cos"]
)
predictions_df["pred_dp_deg"] = angle_from_sin_cos(
    predictions_df["pred_dp_sin"], predictions_df["pred_dp_cos"]
)
predictions_df["dir_error_deg"] = circular_error_deg(
    predictions_df["pred_dir_deg"], predictions_df["target_dir_deg"], absolute=True
)
predictions_df["dp_error_deg"] = circular_error_deg(
    predictions_df["pred_dp_deg"], predictions_df["target_dp_deg"], absolute=True
)

resolved_results_dir = Path(RESULTS_DIR)
if not resolved_results_dir.is_absolute():
    resolved_results_dir = (REPO_ROOT / resolved_results_dir).resolve()

print(f"Resolved repo root: {REPO_ROOT}")
print(f"Resolved results dir: {resolved_results_dir}")
print(f"Loaded split: {SPLIT}")
print(f"Prediction rows: {len(predictions_df):,}")
print(f"Unique sites: {predictions_df['site'].astype(str).nunique():,}")

assert predictions_df["dir_error_deg"].dropna().between(0, 180).all(), (
    "Circular direction errors must lie in [0, 180]."
)
assert predictions_df["dp_error_deg"].dropna().between(0, 180).all(), (
    "Circular Dp errors must lie in [0, 180]."
)

show_note(
    f"Loaded **`predictions_{SPLIT}.csv`**-compatible artifacts from **`{resolved_results_dir}`** and reconstructed physical direction angles from sine/cosine components."
)

predictions_df.head()

In [ ]:
site_metric_rows = []
for site_name, site_df in predictions_df.groupby("site", sort=True):
    row = {
        "site": str(site_name),
        "sample_count": int(len(site_df)),
        "hs_rmse": rmse(site_df["target_hs"], site_df["pred_hs"]),
        "tp_rmse": rmse(site_df["target_tp"], site_df["pred_tp"]),
        "dir_rmse_deg": circular_rmse_deg(site_df["pred_dir_deg"], site_df["target_dir_deg"]),
        "dp_rmse_deg": circular_rmse_deg(site_df["pred_dp_deg"], site_df["target_dp_deg"]),
    }
    site_metric_rows.append(row)

site_metrics_df = pd.DataFrame(site_metric_rows).sort_values("site").reset_index(drop=True)

assert len(site_metrics_df) == predictions_df["site"].astype(str).nunique(), (
    "Expected exactly one metric row per site."
)
assert (site_metrics_df[["hs_rmse", "tp_rmse"]] >= 0).all().all(), (
    "Scalar RMSE values must be non-negative."
)
assert site_metrics_df["dir_rmse_deg"].dropna().between(0, 180).all(), (
    "Dir RMSE must lie in [0, 180]."
)
assert site_metrics_df["dp_rmse_deg"].dropna().between(0, 180).all(), (
    "Dp RMSE must lie in [0, 180]."
)

site_metrics_df.round(4)

In [ ]:
def ranked_metric_table(df: pd.DataFrame, metric_col: str, top_n: int) -> pd.DataFrame:
    cols = ["site", "sample_count", metric_col]
    return (
        df[cols]
        .dropna(subset=[metric_col])
        .sort_values(metric_col, ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


worst_hs_sites = ranked_metric_table(site_metrics_df, "hs_rmse", TOP_N_SITES)
worst_tp_sites = ranked_metric_table(site_metrics_df, "tp_rmse", TOP_N_SITES)
worst_dir_sites = ranked_metric_table(site_metrics_df, "dir_rmse_deg", TOP_N_SITES)
worst_dp_sites = ranked_metric_table(site_metrics_df, "dp_rmse_deg", TOP_N_SITES)

show_note("## Worst Sites by Variable")
show_note(f"Top **{TOP_N_SITES}** sites ranked from worst to best within each target variable.")

display(Markdown("### Hs worst sites"))
display(worst_hs_sites.round(4))

display(Markdown("### Tp worst sites"))
display(worst_tp_sites.round(4))

display(Markdown("### Dir worst sites"))
display(worst_dir_sites.round(4))

display(Markdown("### Dp worst sites"))
display(worst_dp_sites.round(4))

In [ ]:
overall_required_cols = ["hs_rmse", "tp_rmse", "dir_rmse_deg", "dp_rmse_deg"]
overall_source_df = site_metrics_df.copy()
missing_overall_sites = (
    overall_source_df[overall_source_df[overall_required_cols].isna().any(axis=1)]["site"]
    .astype(str)
    .tolist()
)
if missing_overall_sites:
    print(
        "Warning: excluding sites with missing RMSE values from overall ranking:",
        ", ".join(missing_overall_sites),
    )

overall_df = overall_source_df.dropna(subset=overall_required_cols).copy()
overall_df["hs_percentile"] = overall_df["hs_rmse"].rank(method="average", pct=True)
overall_df["tp_percentile"] = overall_df["tp_rmse"].rank(method="average", pct=True)
overall_df["dir_percentile"] = overall_df["dir_rmse_deg"].rank(method="average", pct=True)
overall_df["dp_percentile"] = overall_df["dp_rmse_deg"].rank(method="average", pct=True)
overall_df["overall_percentile_mean"] = overall_df[
    ["hs_percentile", "tp_percentile", "dir_percentile", "dp_percentile"]
].mean(axis=1)

worst_overall_sites = (
    overall_df[
        [
            "site",
            "sample_count",
            "hs_rmse",
            "tp_rmse",
            "dir_rmse_deg",
            "dp_rmse_deg",
            "hs_percentile",
            "tp_percentile",
            "dir_percentile",
            "dp_percentile",
            "overall_percentile_mean",
        ]
    ]
    .sort_values("overall_percentile_mean", ascending=False)
    .head(TOP_N_SITES)
    .reset_index(drop=True)
)

assert worst_overall_sites["overall_percentile_mean"].is_monotonic_decreasing, (
    "Overall ranking must be sorted descending."
)

show_note("## Worst Sites Overall")
show_note(
    "Overall ranking uses percentile-normalized RMSE columns so that `hs`, `tp`, `dir`, and `dp` contribute comparably despite different units."
)

worst_overall_sites.round(4)

In [ ]:
summary_df = load_optional_target_metrics_summary(RESULTS_DIR)
comparison_df = None
if summary_df is None:
    show_note(
        "> `target_metrics_summary.csv` was not found in this results folder, so no artifact cross-check was run."
    )
else:
    summary_site_df = summary_df[
        (summary_df["split"] == SPLIT) & (summary_df["aggregation"] == "site")
    ].copy()
    if summary_site_df.empty:
        show_note(
            "> `target_metrics_summary.csv` exists, but it does not contain per-site rows for the selected split."
        )
    else:
        artifact_metrics_df = (
            summary_site_df.pivot(index="site", columns="target", values="rmse")
            .rename(
                columns={
                    "hs": "hs_rmse_artifact",
                    "tp": "tp_rmse_artifact",
                    "dir": "dir_rmse_deg_artifact",
                    "dp": "dp_rmse_deg_artifact",
                }
            )
            .reset_index()
        )
        comparison_df = site_metrics_df.merge(artifact_metrics_df, on="site", how="inner")
        comparison_df["hs_rmse_abs_diff"] = (
            comparison_df["hs_rmse"] - comparison_df["hs_rmse_artifact"]
        ).abs()
        comparison_df["tp_rmse_abs_diff"] = (
            comparison_df["tp_rmse"] - comparison_df["tp_rmse_artifact"]
        ).abs()
        comparison_df["dir_rmse_abs_diff"] = (
            comparison_df["dir_rmse_deg"] - comparison_df["dir_rmse_deg_artifact"]
        ).abs()
        comparison_df["dp_rmse_abs_diff"] = (
            comparison_df["dp_rmse_deg"] - comparison_df["dp_rmse_deg_artifact"]
        ).abs()

        max_diffs = comparison_df[
            ["hs_rmse_abs_diff", "tp_rmse_abs_diff", "dir_rmse_abs_diff", "dp_rmse_abs_diff"]
        ].max()
        print("Cross-check against target_metrics_summary.csv")
        print(max_diffs.to_string())

        display(Markdown("### Sample cross-check rows"))
        display(
            comparison_df[
                [
                    "site",
                    "hs_rmse",
                    "hs_rmse_artifact",
                    "tp_rmse",
                    "tp_rmse_artifact",
                    "dir_rmse_deg",
                    "dir_rmse_deg_artifact",
                    "dp_rmse_deg",
                    "dp_rmse_deg_artifact",
                ]
            ]
            .head(10)
            .round(4)
        )

In [ ]:
show_note("## Full Per-Site Metrics Table")
full_metrics_display = (
    overall_df[
        [
            "site",
            "sample_count",
            "hs_rmse",
            "tp_rmse",
            "dir_rmse_deg",
            "dp_rmse_deg",
            "hs_percentile",
            "tp_percentile",
            "dir_percentile",
            "dp_percentile",
            "overall_percentile_mean",
        ]
    ]
    .sort_values("overall_percentile_mean", ascending=False)
    .reset_index(drop=True)
)
full_metrics_display.round(4)